# Phase 3 v2: Re-Predict 250k with Improved 90% Model

**Goal:** Update the 238k predictions using the better model (trained on updated 4,500 with 41 fixes)

**Improvements over Phase 3 v1:**
- ✅ Trained on updated `annotation_cleaned.xlsx` (with 41 re-review fixes)
- ✅ Uses **Multi-Seed Ensemble** (5 models) for final predictions
- ✅ Expected accuracy: ~90% (vs 88.36% in v1)
- ✅ Better pseudo-labels for downstream use

**Files needed:**
1. `annotation_cleaned.xlsx` (updated with 41 fixes)
2. `spice_disease_associations.csv` (250,848 rows to predict)

**Time:** ~2 hours on Kaggle T4×2

## Setup
1. New Kaggle notebook → File → Import → upload this `.ipynb`
2. Add Data → Upload both files above
3. Settings → GPU T4×2 + Internet ON
4. Run All

## Step 1: Setup

In [ ]:
!pip install -q openpyxl

import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 2: Find Files

In [ ]:
import os

annotation_path = None
predict_path = None

for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        full = os.path.join(root, f)
        if 'annotation_cleaned' in f.lower() and f.endswith('.xlsx'):
            annotation_path = full
        elif 'spice_disease_associations' in f.lower() and f.endswith('.csv'):
            predict_path = full

print(f'Training: {annotation_path}')
print(f'To predict: {predict_path}')

if not annotation_path or not predict_path:
    raise FileNotFoundError('Upload both annotation_cleaned.xlsx AND spice_disease_associations.csv')

## Step 3: Load Data

In [ ]:
import pandas as pd
import numpy as np

# Training data
train_df = pd.read_excel(annotation_path)
train_df = train_df.dropna(subset=['Info (English)', 'Association Nature'])
train_df = train_df[train_df['Association Nature'].str.strip() != '']

train_df['text'] = ('[SPICE] ' + train_df['Spice'].astype(str).str.strip() + 
                     ' [DISEASE] ' + train_df['Disease'].astype(str).str.strip() + 
                     ' [INFO] ' + train_df['Info (English)'].astype(str).str.strip())
train_df['label'] = train_df['Association Nature'].astype(str).str.strip().str.lower()

print(f'Training rows: {len(train_df)}')
print('Distribution:')
for label, count in train_df['label'].value_counts().items():
    print(f'  {label}: {count} ({count/len(train_df)*100:.1f}%)')

X_train = train_df['text'].values
y_train = train_df['label'].values

# Prediction target (250k)
predict_df = pd.read_csv(predict_path)
print(f'\nTo predict: {len(predict_df)} rows')

# Handle column naming (Title column may or may not exist)
text_col = 'Info (English)' if 'Info (English)' in predict_df.columns else 'evidence_sentence'
spice_col = 'Spice' if 'Spice' in predict_df.columns else 'spice'
disease_col = 'Disease' if 'Disease' in predict_df.columns else 'disease'
title_col = 'Title' if 'Title' in predict_df.columns else 'title' if 'title' in predict_df.columns else None

if title_col:
    predict_df[text_col] = predict_df[text_col].fillna(predict_df[title_col])
predict_df = predict_df[predict_df[text_col].astype(str).str.strip() != '']
predict_df = predict_df[predict_df[text_col].astype(str).str.strip().str.lower() != 'nan']

predict_df['text'] = ('[SPICE] ' + predict_df[spice_col].astype(str).str.strip() + 
                       ' [DISEASE] ' + predict_df[disease_col].astype(str).str.strip() + 
                       ' [INFO] ' + predict_df[text_col].astype(str).str.strip())

print(f'Valid rows to predict: {len(predict_df)}')

## Step 4: Setup

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from collections import Counter
import random
import warnings
warnings.filterwarnings('ignore')

MODEL_NAME  = 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract'
MAX_LEN     = 256
BATCH_SIZE  = 16
EPOCHS      = 5
LR          = 2e-5
LABEL2ID    = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL    = {0: 'negative', 1: 'neutral', 2: 'positive'}
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SEEDS = [42, 123, 456, 789, 999]  # Multi-seed for ensemble

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = [LABEL2ID[l] for l in labels] if labels is not None else [0]*len(texts)
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                              padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long),
        }

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Ready to train ensemble')

## Step 5: Train 5 Models + Predict on 250k

In [ ]:
import time

# Preload prediction data (shared across all models)
predict_ds = TextDataset(predict_df['text'].values, None, tokenizer, MAX_LEN)
predict_loader = DataLoader(predict_ds, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2)

all_ensemble_probs = np.zeros((len(predict_df), 3))

for seed_num, seed in enumerate(SEEDS, 1):
    print(f'\n{"="*55}')
    print(f'  Model {seed_num}/{len(SEEDS)} (seed={seed})')
    print(f'{"="*55}')
    set_seed(seed)
    
    # Train
    train_ds = TextDataset(X_train, y_train, tokenizer, MAX_LEN)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID,
    ).to(DEVICE)
    
    counts = Counter(y_train)
    total = len(y_train)
    weights = torch.tensor(
        [total/(3*counts['negative']),
         total/(3*counts['neutral']),
         total/(3*counts['positive'])],
        dtype=torch.float
    ).to(DEVICE)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, total_steps//10, total_steps)
    
    start = time.time()
    for epoch in range(1, EPOCHS+1):
        model.train()
        loss_sum = 0
        for batch in train_loader:
            optimizer.zero_grad()
            outputs = model(input_ids=batch['input_ids'].to(DEVICE),
                            attention_mask=batch['attention_mask'].to(DEVICE))
            loss = loss_fn(outputs.logits, batch['label'].to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            loss_sum += loss.item()
        print(f'  Epoch {epoch}/{EPOCHS}: loss={loss_sum/len(train_loader):.4f}')
    print(f'  Training time: {(time.time()-start)/60:.1f} min')
    
    # Predict
    print(f'  Predicting on {len(predict_df)} rows...')
    start = time.time()
    model.eval()
    seed_probs = []
    with torch.no_grad():
        for i, batch in enumerate(predict_loader):
            outputs = model(input_ids=batch['input_ids'].to(DEVICE),
                            attention_mask=batch['attention_mask'].to(DEVICE))
            probs = torch.softmax(outputs.logits, dim=1)
            seed_probs.append(probs.cpu().numpy())
            if (i+1) % 200 == 0:
                print(f'    Batch {i+1}/{len(predict_loader)}')
    seed_probs = np.concatenate(seed_probs)
    print(f'  Prediction time: {(time.time()-start)/60:.1f} min')
    
    # Add to ensemble
    all_ensemble_probs += seed_probs / len(SEEDS)
    
    del model
    torch.cuda.empty_cache()

print(f'\n✓ All {len(SEEDS)} models trained and predictions ensembled')

## Step 6: Add Predictions + Filter by Confidence

In [ ]:
predict_df = predict_df.reset_index(drop=True)
predict_preds = np.argmax(all_ensemble_probs, axis=1)

# Update Association Nature column with new predictions
predict_df['Association Nature'] = [ID2LABEL[p] for p in predict_preds]
predict_df['P_negative'] = all_ensemble_probs[:, 0].round(4)
predict_df['P_neutral']  = all_ensemble_probs[:, 1].round(4)
predict_df['P_positive'] = all_ensemble_probs[:, 2].round(4)
predict_df['max_confidence'] = all_ensemble_probs.max(axis=1).round(4)

predict_df = predict_df.drop(columns=['text'], errors='ignore')

# Distribution
print('New prediction distribution (all 250k):')
for label, count in predict_df['Association Nature'].value_counts().items():
    print(f'  {label}: {count:,} ({count/len(predict_df)*100:.1f}%)')

# Confidence filter
THRESHOLD = 0.75
high_conf = predict_df[predict_df['max_confidence'] >= THRESHOLD].copy()
low_conf  = predict_df[predict_df['max_confidence'] <  THRESHOLD].copy()

print(f'\nAt threshold {THRESHOLD}:')
print(f'  High confidence: {len(high_conf):,} ({len(high_conf)/len(predict_df)*100:.1f}%)')
print(f'  Low confidence:  {len(low_conf):,} ({len(low_conf)/len(predict_df)*100:.1f}%)')

## Step 7: Save & Download

In [ ]:
predict_df.to_csv('/kaggle/working/all_predictions_v2.csv', index=False)
high_conf.to_csv('/kaggle/working/final_database_v2.csv', index=False)
low_conf.to_csv('/kaggle/working/uncertain_predictions_v2.csv', index=False)

with open('/kaggle/working/phase3_v2_summary.txt', 'w') as f:
    f.write('='*65 + '\n')
    f.write('  PHASE 3 v2 — Re-Predicted 250k with 90% Model\n')
    f.write('='*65 + '\n\n')
    f.write(f'Model: PubMedBERT Multi-Seed Ensemble (5 seeds)\n')
    f.write(f'Training: 4,500 manual labels (with 41 re-review fixes)\n')
    f.write(f'Model accuracy (from Multi-Seed 80/20): 90.00%\n\n')
    f.write(f'Total predicted: {len(predict_df):,}\n')
    f.write(f'High confidence (>={THRESHOLD}): {len(high_conf):,} ({len(high_conf)/len(predict_df)*100:.1f}%)\n')
    f.write(f'Low confidence (flagged): {len(low_conf):,} ({len(low_conf)/len(predict_df)*100:.1f}%)\n\n')
    f.write('High-confidence label distribution:\n')
    for label, count in high_conf['Association Nature'].value_counts().items():
        f.write(f'  {label}: {count:,} ({count/len(high_conf)*100:.1f}%)\n')
    f.write('\n' + '='*65 + '\n')

print('✓ Files saved:')
print(f'  all_predictions_v2.csv       — {len(predict_df):,} predictions')
print(f'  final_database_v2.csv        — {len(high_conf):,} high-confidence')
print(f'  uncertain_predictions_v2.csv — {len(low_conf):,} flagged')
print(f'  phase3_v2_summary.txt        — summary')

from google.colab import files as _f
# Or Kaggle download: files appear in /kaggle/working/ automatically